# Fills by trade type
Reads `out/fills.csv` (run `python3 main.py` first) and shows the top 20 rows per venue.

In [ ]:
import pandas as pd

pd.set_option("display.width", 160)
fills = pd.read_csv("out/fills.csv")
print(fills.shape)
fills.groupby(["capacity", "venue"])["quantity"].agg(fills="count", shares="sum")

In [ ]:
# INTERNAL — principal fills (firm is the counterparty)
fills[fills["venue"] == "INTERNAL"].head(20)

In [ ]:
# CROSS — client-to-client agency fills (two rows per cross, same price/time)
fills[fills["venue"] == "CROSS"].head(20)

In [ ]:
# MARKET — routed out at the touch (agency)
fills[fills["venue"] == "MARKET"].head(20)

## Unfilled orders — cancelled IOC and expired DAY
Both classes of unfilled orders, with the NBBO and the firm's principal position at arrival. IOC: cancelled immediately (non-marketable on arrival). DAY: rested all day and expired at the 16:00 close (still non-marketable at the final quote — routing them would violate their limit).

In [ ]:
import config
from internalizer.data import load_orders, load_quotes

quotes = load_quotes(config.QUOTES_CSV)
orders = load_orders(config.ORDERS_CSV)
firm = pd.read_csv(config.FIRM_TRADES_CSV, parse_dates=["timestamp"])

# quote tape as a frame (prices back to dollars)
qdf = pd.DataFrame({"ts": [q.ts for q in quotes],
                    "bid": [q.bid / 100 for q in quotes],
                    "ask": [q.ask / 100 for q in quotes]})
last_q = qdf.iloc[-1]   # prevailing NBBO at the close

filled = fills.groupby("order_id")["quantity"].sum()

# principal position timeline: principal fills (client buy = firm sells) + firm hedges
pf = fills[fills["capacity"] == "PRINCIPAL"].copy()
pf["ts"] = pd.to_datetime(pf["timestamp"])
pf["dpos"] = pf["quantity"].where(pf["side"] == "SELL", -pf["quantity"])
fh = firm.rename(columns={"timestamp": "ts"})[["ts", "side", "quantity"]].copy()
fh["dpos"] = fh["quantity"].where(fh["side"] == "BUY", -fh["quantity"])
tl = pd.concat([pf[["ts", "dpos"]], fh[["ts", "dpos"]]]).sort_values("ts")
tl["pos"] = tl["dpos"].cumsum()


def unfilled_table(tif):
    """Orders of the given TIF that did not fully fill, with arrival NBBO and position."""
    rows = []
    for o in orders:
        if o.tif != tif or filled.get(o.order_id, 0) >= o.quantity:
            continue
        q = qdf[qdf["ts"] <= o.ts].iloc[-1]              # prevailing NBBO at arrival
        p = tl[tl["ts"] <= o.ts]
        pos = int(p["pos"].iloc[-1]) if len(p) else 0
        limit = o.limit / 100
        # distance from the touch: at arrival for IOC, at the close for expired DAY
        ref = q if tif == "IOC" else last_q
        away_c = round((ref["ask"] - limit) * 100) if o.side == "BUY" else round((limit - ref["bid"]) * 100)
        when = "at arrival" if tif == "IOC" else "at close"
        rows.append({
            "order_id": o.order_id, "time": o.ts.strftime("%H:%M:%S.%f")[:-3],
            "client": o.client_id.replace("CLIENT_", ""), "side": o.side,
            "qty": o.quantity, "filled": int(filled.get(o.order_id, 0)), "limit": limit,
            "bid": q["bid"], "ask": q["ask"], "spread_c": round((q["ask"] - q["bid"]) * 100),
            "principal_pos": pos,
            "reason": ("CANCELLED: " if tif == "IOC" else "EXPIRED: ")
                      + f"non-marketable, {away_c}c away from the touch {when}",
        })
    return pd.DataFrame(rows)


unfilled_ioc = unfilled_table("IOC")
print(f"{len(unfilled_ioc)} cancelled IOC orders, {unfilled_ioc['qty'].sum():,} shares")
unfilled_ioc

In [ ]:
# Expired DAY orders — rested all day, still non-marketable at the final quote
unfilled_day = unfilled_table("DAY")
print(f"{len(unfilled_day)} expired DAY orders, "
      f"{(unfilled_day['qty'] - unfilled_day['filled']).sum():,} shares expired")
unfilled_day

## Version comparison — v0 / v0.1 / v0.2 / v0.3
Replays the day under all four policies and compares P&L attribution, hedging, and exposure.
- **v0**: new risk sized against the hard limit only; soft-band breaches hedged at the touch.
- **v0.1** (`scan_resting_book`): new risk sized to room-to-the-soft-limit; the residual routes instead of being hedged at the touch.
- **v0.2** (`bleed_hedging`): v0.1 + proactive bleed to 4,000 sh on cheap spreads (≤1¢) or 10-minute aged inventory.
- **v0.3** (`midpoint_offer`): v0.2 + non-marketable limits resting inside the spread are internalized when the improved price meets their limit.

In [ ]:
from dataclasses import replace
from datetime import datetime

from internalizer.engine import Engine
from internalizer.reporting import Reporter
from internalizer.strategy import Strategy
from main import merge_events

_quotes = load_quotes(config.QUOTES_CSV)
NO_BLEED = replace(config.STRATEGY, bleed_trigger=10**9)   # disables the v0.2 bleed


class V0Strategy(Strategy):
    """v0 sizing: new risk capped by the hard limit only (no soft-limit gate)."""
    def principal_quote(self, order, position, ts, bid, ask):
        spread = ask - bid
        buy = order.side == "BUY"
        if (spread >= self.cfg.min_internalize_spread
                and ts.time() < self.cfg.no_new_risk_after):
            cap = (position + self.cfg.hard_position_limit if buy
                   else self.cfg.hard_position_limit - position)
            qty = min(order.remaining, cap)
            if qty > 0:
                return qty, self.improved_price(order.side, bid, ask)
        return super().principal_quote(order, position, ts, bid, ask)


class PreV03Engine(Engine):
    """v0 through v0.2: a non-marketable order goes straight to the book."""
    def _offer_midpoint(self, o, ts):
        return


def run_version(strategy, engine_cls=PreV03Engine):
    _orders = load_orders(config.ORDERS_CSV)
    rep = Reporter()
    eng = engine_cls(strategy, rep)
    for kind, ev in merge_events(_quotes, _orders):
        (eng.on_quote if kind == "Q" else eng.on_order)(ev)
    eng.on_close()

    # P&L attribution + exposure from the trade stream
    trades = []
    for f in rep.fills:
        if f["capacity"] != "PRINCIPAL":
            continue
        d = -f["quantity"] if f["side"] == "BUY" else f["quantity"]
        trades.append((datetime.fromisoformat(f["timestamp"]), d, f["_px"],
                       (f["_bid"] + f["_ask"]) / 2, "fill"))
    for t in rep.firm_trades:
        d = t["quantity"] if t["side"] == "BUY" else -t["quantity"]
        px = round(float(t["price"]) * 100)
        mid = (round(float(t["nbbo_bid"]) * 100) + round(float(t["nbbo_ask"]) * 100)) / 2
        trades.append((datetime.fromisoformat(t["timestamp"]), d, px, mid, "hedge"))
    trades.sort(key=lambda x: x[0])

    edge_f = edge_h = drift = tw = 0.0
    pos, prev_mid, prev_ts, mx = 0, None, None, 0
    for ts, d, px, mid, kind in trades:
        if prev_mid is not None:
            drift += pos * (mid - prev_mid)
            tw += abs(pos) * (ts - prev_ts).total_seconds()
        e = (mid - px) * d
        edge_f, edge_h = edge_f + (e if kind == "fill" else 0), edge_h + (e if kind == "hedge" else 0)
        pos += d
        mx = max(mx, abs(pos))
        prev_mid, prev_ts = mid, ts
    span = (trades[-1][0] - trades[0][0]).total_seconds()

    hshares = sum(t["quantity"] for t in rep.firm_trades)
    return {
        "P&L ($)": round(eng.cash / 100, 2),
        "edge: client fills ($)": round(edge_f / 100, 2),
        "edge: hedges ($)": round(edge_h / 100, 2),
        "inventory drift ($)": round(drift / 100, 2),
        "hedge trades": len(rep.firm_trades),
        "hedge shares": hshares,
        "avg hedge cost (c/sh)": round(-edge_h / hshares, 2) if hshares else 0.0,
        "internalized shares": sum(f["quantity"] for f in rep.fills if f["venue"] == "INTERNAL"),
        "routed shares": sum(f["quantity"] for f in rep.fills if f["venue"] == "MARKET"),
        "client improvement ($)": round(sum(
            (f["_ask"] - f["_px"] if f["side"] == "BUY" else f["_px"] - f["_bid"]) * f["quantity"]
            for f in rep.fills if f["venue"] in ("INTERNAL", "CROSS")) / 100, 2),
        "max |pos| (sh)": mx,
        "time-wtd avg |pos| (sh)": round(tw / span),
        "EOD position": eng.position,
    }


comparison = pd.DataFrame({
    "v0": run_version(V0Strategy(NO_BLEED)),
    "v0.1 scan_resting_book": run_version(Strategy(NO_BLEED)),
    "v0.2 bleed_hedging": run_version(Strategy(config.STRATEGY)),
    "v0.3 midpoint_offer": run_version(Strategy(config.STRATEGY), Engine),
})
comparison

## P&L attribution — derivation
Every principal trade is marked against the **contemporaneous mid**. Writing each trade's price as `mid ± deviation`, the cash P&L telescopes into an exact identity (no residual, because the firm ends flat):

$$\text{P\&L} \;=\; \underbrace{\sum_i (mid_i - px_i)\cdot q_i}_{\text{execution edge}} \;+\; \underbrace{\sum_i pos_i\cdot(mid_{i+1}-mid_i)}_{\text{inventory drift}}$$

- **edge**: what each trade earned relative to fair value at that instant (internalize ≈ +0–0.5¢, touch reduce-fill = +half-spread, hedge = −half-spread)
- **drift**: what the inventory earned or lost as the mid moved between trades

The cell below derives this line by line and cross-checks that `edge + drift == engine cash` for all three versions.

In [ ]:
def attribute(strategy, engine_cls=PreV03Engine):
    """Replay the day under `strategy`, decompose principal P&L into edge + drift."""
    rep = Reporter()                                        # fresh fill/hedge recorder
    eng = engine_cls(strategy, rep)                         # fresh engine, flat book, zero cash
    for kind, ev in merge_events(_quotes, load_orders(config.ORDERS_CSV)):  # one time-ordered stream
        (eng.on_quote if kind == "Q" else eng.on_order)(ev)  # dispatch quotes/orders in arrival order
    eng.on_close()                                          # 16:00: final sweep, flatten, expire

    trades = []                                             # unified stream of the firm's principal trades
    for f in rep.fills:                                     # client fills first...
        if f["capacity"] != "PRINCIPAL":                    # cross/route legs carry no firm risk
            continue                                        # ...so they are excluded from attribution
        signed = -f["quantity"] if f["side"] == "BUY" else f["quantity"]  # client BUY = firm SELLS (negative)
        mid = (f["_bid"] + f["_ask"]) / 2                   # fair value = mid of the NBBO recorded on the fill
        trades.append((datetime.fromisoformat(f["timestamp"]), signed, f["_px"], mid))  # (time, firm qty, price, mid)
    for t in rep.firm_trades:                               # ...then the firm's own hedges
        signed = t["quantity"] if t["side"] == "BUY" else -t["quantity"]  # firm BUY adds to position
        px = round(float(t["price"]) * 100)                 # blotter stores dollars; convert to cents
        mid = (round(float(t["nbbo_bid"]) * 100) + round(float(t["nbbo_ask"]) * 100)) / 2  # mid at hedge time
        trades.append((datetime.fromisoformat(t["timestamp"]), signed, px, mid))  # same tuple shape
    trades.sort(key=lambda x: x[0])                         # chronological: drift accrues between consecutive trades

    edge = drift = 0.0                                      # the two attribution buckets (cents x shares)
    pos = 0                                                 # firm position, rebuilt trade by trade
    prev_mid = None                                         # mid at the previous trade (for the drift term)
    for ts, signed, px, mid in trades:                      # walk the trade stream once
        if prev_mid is not None:                            # no drift before the first trade
            drift += pos * (mid - prev_mid)                 # inventory P&L: position x mid move since last trade
        edge += (mid - px) * signed                         # trade P&L vs fair value: buy below mid = +, sell above = +
        pos += signed                                       # update position after the trade
        prev_mid = mid                                      # this mid becomes the next interval's start
    assert pos == 0                                         # firm must end flat (close-flatten guarantees it)
    assert round(edge + drift) == eng.cash                  # the identity: attribution reproduces cash exactly

    return {"edge ($)": round(edge / 100, 2),               # execution edge in dollars
            "drift ($)": round(drift / 100, 2),             # inventory drift in dollars
            "total = edge + drift ($)": round((edge + drift) / 100, 2),  # their sum...
            "engine cash ($)": round(eng.cash / 100, 2)}    # ...must equal the engine's cash (cross-check row)


attrib = pd.DataFrame({                                     # run the derivation for all four versions
    "v0": attribute(V0Strategy(NO_BLEED)),                  # hard-limit sizing, forced hedges
    "v0.1 scan_resting_book": attribute(Strategy(NO_BLEED)),  # soft-limit gating, no bleed
    "v0.2 bleed_hedging": attribute(Strategy(config.STRATEGY)),  # + dual-trigger proactive bleed
    "v0.3 midpoint_offer": attribute(Strategy(config.STRATEGY), Engine),  # + inside-spread limits
})
attrib                                                      # edge row moves with policy; drift row is the path